In [4]:
print("hello")

hello


In [5]:
from os import path
main_path='t2-ragbench/data/ConvFinQA'

json_path=path.join(main_path,'turn_0.jsonl')


In [6]:
import json

def Load_josn(p):
    with open(p, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return d


pdf_urls=Load_josn('data_set\\pdf_urls.json')
question_doc = Load_josn('data_set\\qrels.json')
queries=Load_josn('data_set\\queries.json')
answers=Load_josn('data_set\\answers.json')



In [7]:
import pandas as pd
df = pd.DataFrame.from_dict(question_doc, orient='index')
df.index.name = 'query_id'
df = df.reset_index()  

print(df)

                                  query_id        doc_id  section_id
0     852703f0-8373-43a2-a18a-eb5908ad0779  2410.14077v2           1
1     9199173b-3ed1-4118-88cd-1713fc5fa8a7  2404.00822v2          17
2     1d585069-a446-47fa-a74d-0387316ea330  2410.07168v2          30
3     dc064d11-cd18-4866-8a99-f16b0abec9c6  2401.07294v4          12
4     283afa84-f0c8-40a7-a6f1-fb2a6b97c761  2411.14884v3           1
...                                    ...           ...         ...
3040  08a34950-7004-433d-ac0e-8c48363ce406  2410.23587v3           2
3041  029ae88b-dc53-4f88-918b-c8478d7ef0d3  2410.12710v2           2
3042  d369250a-8506-4c6c-948b-e9383e88e0e2  2410.10516v3          15
3043  6d7c948c-4d17-4b27-bb83-eb2b88728035  2408.02322v2           0
3044  90144eed-61dd-4a4a-b85d-0e01d597004a  2403.12117v2          27

[3045 rows x 3 columns]


In [8]:
doc_list=df.groupby('doc_id').count().sort_values(by='query_id',ascending=False).head(10).index.tolist()
df=df[df['doc_id'].isin(doc_list)]

In [ ]:
import requests, os
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs("pdfs", exist_ok=True)
items ={}
for i ,j in pdf_urls.items():
    if(i in doc_list):
        items[i]=j
    
def download(name_url):
    name, url = name_url
    try:
        fname = os.path.join("pdfs", f"{name}.pdf")
        
        res = requests.get(url, timeout=30)
        res.raise_for_status()
        with open(fname, "wb") as f:
            f.write(res.content)
        return name, True, None
    except Exception as e:
        return name, False, str(e)

pdfs_ = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(download, item) for item in items.items()]
    for future in as_completed(futures):
        name, ok, err = future.result()
        if ok:
            pdfs_.append(name)
        else:
            print(f"failed: {name} -> {err}")

print(f"{len(pdfs_)}/{len(items)} downloaded")

In [9]:
q_df = pd.DataFrame.from_dict(queries, orient='index')  # columns: query, type, source
q_df.index.name = 'query_id'
q_df = q_df.reset_index().rename(columns={'type': 'query_type', 'source': 'query_source', 'query': 'query'})

df = df.merge(q_df, on='query_id', how='left')

In [10]:
answer_df=pd.DataFrame.from_dict(answers,orient="index")
answer_df.index.name='query_id'
df=df.merge(answer_df,on='query_id',how='left')

In [11]:
df=df.rename(columns={0:'answer'})


In [12]:
df.head(2)

,query_id,doc_id,section_id,query,query_type,query_source,answer
0,dc064d11-cd18-4866-8a99-f16b0abec9c6,2401.07294v4,12,How does the MLMM approach affect the analysis...,abstractive,text-image,The MLMM approach affects the analysis of RMSE...
1,bac61451-d99a-43b3-9754-b8a593e5d1d7,2401.11899v3,10,Does bounded invariance affect how probability...,extractive,text,"No, bounded invariance states that changes in ..."


In [13]:
from rag_eval.parsers import PDFPlumberParser 

In [14]:
p=PDFPlumberParser()

In [15]:
d=p.parse('D:\\projects\\RAG-Chunking-Eval\\pdfs\\2401.03305v2.pdf')

In [16]:
d.pages[0].content

'Leveraging IS and TC: Optimal order execution subject to\nreference strategies\nXue Cheng1, Peng Guo1, and Tai-Ho Wang2\n1\nLMEQF,DepartmentofFinancialMathematics,SchoolofMathematicalSciences,PekingUniversity,Beijing100871,China.\n2DepartmentofMathematics,BaruchCollege,CUNY,1BernardBaruchWay,NewYork,NY10010,USA\nMarch 5, 2025\nAbstract\nThe paper addresses the problem of meta order execution from a broker-dealer’s point of view in\nAlmgren-Chriss model under execution risk. A broker-dealer agency is authorized to execute an order\nof trading on some client’s behalf. The strategies that the agent is allowed to deploy is subject to a\nbenchmark, referred to as the reference strategy, regulated by the client. We formulate the broker’s\nproblem as a utility maximization problem in which the broker seeks to maximize his utility of excess\nprofit-and-lossattheexecutionhorizon,ofwhichoptimalfeedbackstrategiesareobtainedinclosedform.\nIn the absence of execution risk, the optimal strategies s

In [17]:
from rag_eval.chunkers import RecursiveChunker

r=RecursiveChunker()
chunks_=r.chunk(d)

In [18]:
chunks_

[Chunk(text='Leveraging IS and TC: Optimal order execution subject to\nreference strategies\nXue Cheng1, Peng Guo1, and Tai-Ho Wang2\n1\nLMEQF,DepartmentofFinancialMathematics,SchoolofMathematicalSciences,PekingUniversity,Beijing100871,China.\n2DepartmentofMathematics,BaruchCollege,CUNY,1BernardBaruchWay,NewYork,NY10010,USA\nMarch 5, 2025\nAbstract\nThe paper addresses the problem of meta order execution from a broker-dealer’s point of view in', chunk_no=0, page_numbers=[1], source_id='D:\\projects\\RAG-Chunking-Eval\\pdfs\\2401.03305v2.pdf', metadata={}),
 Chunk(text='Almgren-Chriss model under execution risk. A broker-dealer agency is authorized to execute an order\nof trading on some client’s behalf. The strategies that the agent is allowed to deploy is subject to a\nbenchmark, referred to as the reference strategy, regulated by the client. We formulate the broker’s\nproblem as a utility maximization problem in which the broker seeks to maximize his utility of excess\nprofit-and-los

In [35]:
q1_df=df[df['doc_id']=='2401.03305v2']

In [36]:
print("hello")

hello


In [38]:

from rag_eval.embedders import HuggingFaceEmbedder
he=HuggingFaceEmbedder()
chunks_df=pd.DataFrame(columns=['doc_id','doc_path','chunk','page_no','chunk_no','embedding'])
chunk_data=[]

for i in chunks_:
    chunk_data.append({'doc_id':i.source_id,'doc_path':'','page_no':i.page_numbers,'chunk':i.text,'chunk_no':i.chunk_no,'embedding':he.embed_query(i.text)})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3484.77it/s]


In [39]:
q1_df["embedding"] = q1_df.apply(
    lambda x: he.embed_query(x["query"]),
    axis=1
)

C:\Users\vishw\AppData\Local\Temp\ipykernel_4564\3761716432.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q1_df["embedding"] = q1_df.apply(


In [40]:
q1_df.columns

Index(['query_id', 'doc_id', 'section_id', 'query', 'query_type',
       'query_source', 'answer', 'embedding'],
      dtype='object')

In [41]:
chunks_df.columns

Index(['doc_id', 'doc_path', 'chunk', 'page_no', 'chunk_no', 'embedding'], dtype='object')

In [44]:
q1_df.head(1)

,query_id,doc_id,section_id,query,query_type,query_source,answer,embedding
6,0920cb6c-229b-4b46-b2ab-834dffea6689,2401.03305v2,2,How do implementation shortfall (IS) and targe...,abstractive,text,Implementation shortfall (IS) orders aim to ex...,"[-0.031133107841014862, 0.07394389063119888, 0..."


In [47]:
chunks_df=pd.DataFrame(chunk_data)

In [50]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Precompute chunk embeddings ONCE outside the function
chunk_embs = np.vstack(chunks_df['embedding'].values)  # More robust than tolist()

def get_top_5_chunks(row):
    """Find top 5 most similar chunks for a query."""
    query_emb = np.array(row['embedding']).reshape(1, -1)
    
    # Cosine similarity: shape (1, num_chunks)
    similarities = cosine_similarity(query_emb, chunk_embs)[0]
    
    # Get indices of top 5
    top_indices = np.argsort(similarities)[-5:][::-1]
    
    # Return as list of dicts with chunk info + similarity score
    return [
        {
            'chunk': chunks_df.iloc[idx]['chunk'],
            'doc_id': chunks_df.iloc[idx]['doc_id'],
            'page_no': chunks_df.iloc[idx]['page_no'],
            'chunk_no': chunks_df.iloc[idx]['chunk_no'],
            'similarity': float(similarities[idx]),
        }
        for idx in top_indices
    ]

q1_df['top_5'] = q1_df.apply(get_top_5_chunks, axis=1)

C:\Users\vishw\AppData\Local\Temp\ipykernel_4564\3931682640.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q1_df['top_5'] = q1_df.apply(get_top_5_chunks, axis=1)


In [66]:
import json

records = []
for _, row in q1_df.iterrows():
    records.append({
        'query_id': str(row['query_id']),
        'query_type': str(row['query_type']),
        'query': str(row['query']),
        'answer': str(row['answer']),
        'top_5': [
            {
                'chunk': str(c['chunk']),
                'doc_id': str(c['doc_id']),
                'page_no': c['page_no'] if not isinstance(c['page_no'], list) else c['page_no'][0],
                'similarity': float(c['similarity']),
            }
            for c in row['top_5']
        ]
    })

# Read the template HTML, replace the DATA constant, write out
with open('rag_eval_query_viewer_v2.html') as f:
    html = f.read()
html = html.replace('const DATA = [', f'const DATA = {json.dumps(records)}; const _ORIG = [')
with open('rag_viewer.html', 'w', encoding='utf-8') as f:
    f.write(html)